<a href="https://colab.research.google.com/github/mybenkhadda/GEMS---drivendata/blob/main/notebooks/colab/GEMS_G05_1_SYSTEM_RECOVERY_AND_RANKING.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GEMS G05.1 - fault-system recovery & intelligent geological ranking (code rev g051.1-r1)

Why did W2 miss the 8 G05 investigation systems, can an equal-budget candidate union recover them, and can a residual ranker on top of frozen W2 turn that into official discovery DTI **without** losing geographic DTI?

* Reuses the submission factory (frozen B0 / B1 / W2 engine), the GeoMoE units / candidate inventory / official scorer / canonical system recovery / frozen submitted-W2 policy / fixed-policy selection.
* Outputs: `MyDrive/GEMS_2/outputs/g05_1_system_recovery/<run_id>/` (deterministic run id; an earlier successful stage output is archived under `history/` before it is replaced).
* **First run** (`RUN` defaults): preflight, atlas, recall, registry, graph. Training, strict audit, selection, ablations, reconstruction and export are opt-in.
* Learned rankers are `SCREENING_ONLY` (cross-fitted, not strictly nested); only label-free S1 can become `PROMOTE_CANDIDATE`. Nothing is uploaded: a validated file is at best `READY_FOR_MANUAL_UPLOAD`.
* The public score (0.0942) and the historical G05 numbers are references measured under other protocols - never tuning targets.

Docs: `docs/g05_1/README.md`, `EXPERIMENT_DESIGN.md`, `VALIDATION_CONTRACT.md`, `ARTIFACT_REGISTRY.md`, `SESSION_HANDOFF.md`.

## 00 - CONFIG and RUN

`CONFIG` = where and how; `RUN` = which stages. `overrides` deep-merge into `configs/g05_1/recovery.yaml`; `geomoe_overrides` into `configs/geomoe/discovery_geomoe.yaml` (candidate machinery; changes the run id).

In [ ]:
CONFIG = {
    'repo': 'mybenkhadda/GEMS---drivendata',
    'ref': 'main',                          # pin the pushed commit SHA for a reproducible run
    'drive_root': '/content/drive/MyDrive/GEMS_2',
    'local_root': '/content/gems_work',
    'run_id': None,                         # None = deterministic id from version / inputs / targets / ref
    'reuse_cached': True,
    'sync_to_drive': True,
    'save_png': True,
    'run_synthetic_tests': True,
    'backfill_missing_g02': False,          # True: compute ONLY the missing G02 unit rasters (frozen G02 config)
    'g05_run_id': None,                     # explicit G05 run; else configs policy latest_complete_manifest
    'overrides': {},                        # e.g. {'displacement': {'max_segments_per_unit': 2000}}
    'geomoe_overrides': {},
    'manual_upload_only': True,             # cannot be disabled
}
RUN = {
    'preflight': True, 'atlas': True, 'recall': True, 'registry': True, 'graph': True,   # first run
    'displacement': False,      # Stage E (CPU heavy: profile search per segment)
    'training': False,          # Stage G (G0-G5 + S1, cross-fitted)
    'strict': False,            # Stage H (needed for any promotion: upstream-honesty gate)
    'selection': None,          # Stage I (None = follows 'training')
    'ablations': False,         # Stage J
    'reconstruction': False,    # Stage K research_score.tif
    'export': False,            # validated candidate (only if Stage I = PROMOTE_CANDIDATE)
}
PRIVATE_REPO = False
EXPECT = {'code_rev': 'g051.1-r1', 'commit': None}

## 01 - Environment (Drive, code at the pinned ref, editable install)

In [ ]:
import base64, json, os, subprocess, sys
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
CODE = '/content/GEMS'
REPO_URL = 'https://github.com/' + CONFIG['repo'] + '.git'

def _git_env():
    if not PRIVATE_REPO:
        return None
    from google.colab import userdata
    hdr = base64.b64encode(('x-access-token:' + (userdata.get('GITHUB_TOKEN') or '')).encode()).decode()
    return {**os.environ, 'GIT_CONFIG_COUNT': '1', 'GIT_CONFIG_KEY_0': 'http.https://github.com/.extraheader',
            'GIT_CONFIG_VALUE_0': 'AUTHORIZATION: basic ' + hdr}

def git(*args, auth=False, check=True):
    r = subprocess.run(['git', *args], capture_output=True, text=True, env=_git_env() if auth else None)
    if check and r.returncode:
        raise RuntimeError('git ' + ' '.join(args[:4]) + ' failed:\n' + r.stderr[-1500:])
    return r

if not Path(CODE, '.git').exists():
    git('clone', '-q', REPO_URL, CODE, auth=True)
git('-C', CODE, 'fetch', '-q', '--tags', 'origin', auth=True)
ref = CONFIG['ref']
if git('-C', CODE, 'rev-parse', '--verify', '-q', 'refs/remotes/origin/' + ref, check=False).returncode == 0:
    git('-C', CODE, 'checkout', '-q', '-B', 'g051_colab_active', 'origin/' + ref)
else:
    git('-C', CODE, 'checkout', '-q', '--detach', ref)
HEAD = git('-C', CODE, 'rev-parse', 'HEAD').stdout.strip()
if 'gems_bench' in sys.modules:
    raise RuntimeError('gems_bench already imported: Runtime -> Restart session, then run from the top')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', CODE + '[colab]', 'pytest'], check=True)
sys.path.insert(0, CODE)
os.environ['GEMS_CODE_COMMIT'] = HEAD
print('HEAD', HEAD)

## 02 - Imports, version check, factory + G05.1 runner

In [ ]:
import pandas as pd
from IPython.display import HTML, Image, Markdown, display
import gems_bench.g05_1 as G51
from gems_bench.g05_1 import runner as RN
from gems_bench.geomoe import run as GR
from gems_bench.leaderboard_focus import factory as FA
assert G51.G051_CODE_REV == EXPECT['code_rev'], f'code rev {G51.G051_CODE_REV} != {EXPECT["code_rev"]}'
assert not EXPECT['commit'] or HEAD.startswith(EXPECT['commit']), f'HEAD {HEAD} != {EXPECT["commit"]}'
F = FA.Factory(GR.factory_config({**GR.NB_DEFAULTS, **{k: CONFIG[k] for k in ('repo', 'ref', 'drive_root', 'local_root', 'reuse_cached', 'sync_to_drive')}}), HEAD)
R = RN.G051(CONFIG, RUN, HEAD, factory=F)
OUT = R.ws.root

def show(rel, cols=None, n=40):
    p = OUT / rel
    if not p.exists():
        print('ABSENT:', rel); return None
    df = pd.read_csv(p) if p.suffix == '.csv' else pd.read_parquet(p)
    display(df[[c for c in (cols or df.columns) if c in df]].head(n)); return df

def js(rel, n=3000):
    d = R.ws.read_json(rel)
    print(json.dumps(d, indent=1, default=str)[:n] if d is not None else f'ABSENT: {rel}')

print('run id', R.run_id, '| RUN', R.run_flags)
print('local', OUT, '| Drive', R.drive / R.out)

## 03 - Synthetic tests (enforced) and frozen engine

`TESTS_FAILED` blocks the candidate inventory and everything after it.

In [ ]:
R.s00_tests()
if R.status['00_tests'] == 'TESTS_FAILED':
    print((OUT / 'audit/pytest_output.txt').read_text()[-4000:])
show('audit/selftests.csv', ['test', 'status', 'detail'])
R.s01_attach()

## 04 - Stage A: preflight / dependency inventory (fails closed)

Grid contract (3730 x 3292, EPSG:32611, 100 m, 19 bands), **exact** band identifiers (`iso_grav_anom` is never resolved to `iso_grav_anom_hg`), frozen W2 maps, G02 rasters, frozen policy, optional G05 run chosen by manifest policy (schema discovered, never assumed).

In [ ]:
R.sA_inventory()
show('preflight/dependency_inventory.csv', n=80)
show('preflight/band_resolution.csv')
js('preflight/g05_run_selection.json', 1500)

## 05 - GeoMoE audit, scenarios (V1 / V2 / V3) and leakage checks

In [ ]:
R.s02_audit(); R.s03_scenarios()
show('validation/scenarios.csv')
L = show('audit/label_leakage_audit.csv', n=0)
if L is not None: display(L.status.value_counts())

## 06 - Candidate inventory (validated per-unit cache)

The expensive step of a first run (CNN / W2 / potential-field / G02 layers -> pools -> segments). Finished units are reused after a disconnect.

In [ ]:
R.s04_candidates()
show('candidates/source_provenance.csv', n=60)

## 07 - Frozen submitted W2 baseline, policy parity, metric equivalence, coverage

In [ ]:
R.s05_oracle()
show('validation/frozen_w2_baseline.csv', ['scenario', 'zone', 'rep', 'frac', 'dti', 'n_recovered', 'n_systems'])
show('audit/baseline_policy_parity.csv', ['unit', 'status', 'grid_status', 'jaccard_vs_factory_evaluation_map'])
show('candidates/oracle_coverage.csv')

## 08 - Stage B: failure atlas of the 8 investigation systems

Zone 0: 471, 492, 531; zone 1: 243, 559; zone 2: 356, 544, 560 (from G05's larger union pool, not a matched budget). Each target gets 2 matched recovered controls (same unit, nearest length), 5 figures, measured observations and a rule-based **HYPOTHESIS** (never a finding). Truth outlines are post-hoc only.

In [ ]:
if RUN['atlas']:
    R.sB_atlas()
    show('atlas/target_systems.csv', ['unit', 'system_id', 'category', 'length_km', 'w2_rank_median', 'inventory_near_share', 'hit_share', 'proposers'])
    p = OUT / 'atlas/ATLAS_SUMMARY.md'
    if p.exists(): display(Markdown(p.read_text()))
else:
    R.not_requested('B_atlas', "RUN['atlas'] is False")

## 09 - Case browser (panels of one system)

In [ ]:
SYSTEM = ('V1:0', 471)            # (unit, system id) to display
for name in ('1_learned', '2_cnn', '3_potential', '4_structural', '5_candidates'):
    p = OUT / f"atlas/panels/{SYSTEM[0].replace(':', '_')}_sys{SYSTEM[1]}_{name}.png"
    if p.exists(): display(Image(str(p)))
print('Full browser:', R.drive / R.out / 'atlas/case_browser.html')

## 10 - Stage C: equal-budget candidate recall

Same inventory, representation and eligible budget for every construction. 0.5 / 1 / 2 % are deployable areas; 5 / 10 % are recall **ceilings** (diagnostic). Unions: max-rank (one budget) and quota (budget / n per member).

In [ ]:
if RUN['recall']:
    R.sC_recall()
    P = show('recall/recall_summary.csv', n=0)
    if P is not None and len(P):
        display(P[(P.scenario == 'V1') & (P.role == 'final')].pivot_table(index='construction', columns='frac', values=['dti', 'n_recovered']).round(4))
    p = OUT / 'recall/recall_by_construction.png'
    if p.exists(): display(Image(str(p)))
else:
    R.not_requested('C_recall', "RUN['recall'] is False")

## 11 - Recovery matrix, new / lost systems vs W2, leave-one-expert-out

In [ ]:
T = show('recall/target_system_recovery.csv', n=0)
if T is not None and len(T):
    display(T[T.frac <= 0.02].pivot_table(index=['unit', 'system_id'], columns=['construction', 'frac'], values='recovered', aggfunc='max').astype(int))
show('recall/new_lost_vs_w2.csv', ['unit', 'construction', 'frac', 'n_new', 'n_lost', 'new_ids', 'lost_ids'], n=40)
show('recall/leave_one_expert_out.csv')

## 12 - Stage D: fault-hypothesis registry (D1-D7)

One row per candidate of every unit; label-free features only (`y_near_truth` is the target, guarded). Dictionary, provenance, availability and geometry (GPKG / GeoParquet).

In [ ]:
if RUN['registry']:
    R.sD_registry()
    show('registry/feature_availability.csv', n=60)
else:
    R.not_requested('D_registry', "RUN['registry'] is False")

## 13 - Stage E: displacement-style evidence (opt-in)

Shifted profile correspondence across the trace vs a same-side negative control. Short traces are `UNRESOLVABLE`. G05's `offset_proxy` was curvature and stays curvature.

In [ ]:
if RUN['displacement']:
    R.sE_displacement()
    show('displacement/control_summary.csv')
else:
    R.not_requested('E_displacement', "RUN['displacement'] is False")

## 14 - Stage F: candidate graph (continuity / branch edges, label-free)

In [ ]:
if RUN['graph']:
    R.sF_graph()
    show('graph/graph_summary.csv')
else:
    R.not_requested('F_graph', "RUN['graph'] is False")

## 15 - Stage G: residual rankers (opt-in)

G0 frozen W2, G1 residual LightGBM, G2 pairwise, G3 gated, G4 region-aware (needs EDA domains), G5 graph, S1 strict label-free blend. Labels: positives / hard negatives / ambiguous (excluded) / PU unlabelled. Cross-fitted on other zones + buffer; lambda on inner zones. **Learned results are SCREENING_ONLY.**

In [ ]:
if RUN['training']:
    R.sG_rankers()
    L = show('rankers/label_summary.csv', n=0)
    if L is not None and len(L): display(L.pivot_table(index='unit', columns='cls', values='n', aggfunc='sum'))
    reg = R.ws.read_json('rankers/model_registry.json') or {}
    display(pd.DataFrame([{'key': k, **{a: v.get(a) for a in ('status', 'lambda', 'lambda_status', 'n_train', 'n_positive', 'strictness')}} for k, v in reg.items()]))
else:
    R.not_requested('G_rankers', "RUN['training'] is False")

## 16 - Stage H: strict-nesting audit (opt-in, required for any promotion)

Learned rankers -> `SCREENING_ONLY`; strict nesting of learned models -> `STRICT_NESTING_BLOCKED` (per-pair W2 / CNN refits listed in the retraining plan, not run here).

In [ ]:
if RUN['strict']:
    R.sH_nesting()
    js('nesting/strictness.json')
    A = show('nesting/nesting_audit.csv', n=0)
    if A is not None and len(A): display(A.pivot_table(index='check', columns='model', values='status', aggfunc='first'))
    show('nesting/retraining_plan.csv')
else:
    R.not_requested('H_nesting', "RUN['strict'] is False")

## 17 - Stage I: DTI-aware selection at matched area (fixed policy)

Top / skeleton / crest / soft corridor / soft confidence / confidence width x 0.5-5 % budgets; partial segments by pixel-W2 tie-break; one fixed-policy rule on development units; reference = frozen submitted W2.

In [ ]:
if R.run_flags['selection']:
    R.sI_selection()
    show('selection/model_summary.csv', ['scenario', 'model', 'dti', 'worst_zone_dti', 'worst_zone', 'protocol', 'params_all'], n=60)
    show('selection/challenger_vs_g0.csv', n=60)
else:
    R.not_requested('I_selection', "RUN['selection'] is False")

## 18 - Promotion gates

>= +0.005 discovery DTI vs frozen W2 (recommend >= +0.010), no zone drop > 0.01, geographic DTI under the exact frozen cell >= -0.005, net extra systems >= 1, provenance + upstream honesty verified, export parity (checked on the written file in Stage K). SCREENING_ONLY models stop at `PROMISING_SCREENING_ONLY`.

In [ ]:
show('selection/promotion_decisions.csv', ['model', 'strictness', 'status', 'disc_dti', 'disc_dti_w2', 'disc_gain', 'geo_delta', 'net_systems', 'reasons'])
js('selection/decision.json', 2500)

## 19 - Stage J: ablations and explainability (opt-in)

In [ ]:
if RUN['ablations']:
    R.sJ_ablation()
    show('ablation/ablation_summary.csv')
    show('ablation/family_importance.csv')
    show('ablation/feature_importance.csv', n=25)
else:
    R.not_requested('J_ablation', "RUN['ablations'] is False")

## 20 - Target-system contributions (post-hoc reading of the atlas systems)

In [ ]:
C = show('ablation/target_system_contributions.csv', n=0)
if C is not None and len(C):
    display(C.pivot_table(index='feature', columns='system_id', values='mean_contrib').round(3))

## 21 - Stage K: full-footprint reconstruction (opt-in)

`reconstruction/research_score_<model>.tif` is ALWAYS a research artifact (tag `ROLE=RESEARCH_NOT_A_SUBMISSION`). A `validated_submission_candidate_*.tif` is written only for a `PROMOTE_CANDIDATE` with `RUN['export']`, after every read-back check. **Never uploaded automatically.**

In [ ]:
if RUN['reconstruction']:
    R.sK_reconstruct()
    js('reconstruction/reconstruction_manifest.json', 1500)
    js('submissions/submission_manifest.json', 1500)
else:
    R.not_requested('K_reconstruct', "RUN['reconstruction'] is False")

## 22 - Stage integrity (which conclusions are allowed)

In [ ]:
R.sY_integrity()
show('audit/stage_integrity.csv', ['stage', 'actual_status', 'missing_outputs', 'failure_reason', 'scientific_conclusion_allowed'], n=40)

## 23 - Report and deliverables

In [ ]:
R.sZ_report()
display(Markdown((OUT / 'reports/G051_REPORT.md').read_text()))
show('reports/DELIVERABLES.csv', n=60)

## 24 - Run history (archived earlier outputs; nothing silently overwritten)

In [ ]:
show('history/index.csv', n=60)
R.final_message()

## 25 - RUN_ALL (alternative to the cells above)

Runs every stage in order with the same `RUN` gates, then integrity and the report. Re-running reuses every validated stage.

In [ ]:
# R.run()